# 📘 Capstone: Enterprise Data Governance & Quality Platform
## Databricks Notebook — Cross-Domain Governance Framework

**Project Brief:**
You are the data governance lead at a company with multiple data domains.
Build a complete governance platform covering:

1. **Data Catalog** — Auto-discover and document all tables
2. **Data Classification** — Identify PII, PHI, financial data
3. **Quality Framework** — Define and enforce quality rules
4. **Access Control** — Role-based permission matrix
5. **Lineage Tracking** — Map data flow source → Gold
6. **PII Masking** — Protect sensitive data
7. **Compliance** — GDPR/HIPAA readiness assessment
8. **Data Contracts** — Producer-consumer agreements
9. **Monitoring** — Quality trends, freshness, alerts
10. **Retention** — Data lifecycle management

**Architecture:**
```
┌─────────────────────────────────────────────────────────────────┐
│                 DATA GOVERNANCE PLATFORM                         │
├─────────────────────────────────────────────────────────────────┤
│  ┌──────────┐  ┌──────────────┐  ┌───────────────┐            │
│  │  Catalog  │  │Classification│  │Quality Engine │            │
│  │(discovery)│  │ (PII/PHI)    │  │(rules + scores)│           │
│  └──────────┘  └──────────────┘  └───────────────┘            │
│  ┌──────────┐  ┌──────────────┐  ┌───────────────┐            │
│  │ Lineage  │  │Access Control│  │  Compliance   │            │
│  │ (DAG)    │  │ (RBAC)       │  │ (GDPR/HIPAA) │            │
│  └──────────┘  └──────────────┘  └───────────────┘            │
│  ┌──────────┐  ┌──────────────┐  ┌───────────────┐            │
│  │Monitoring│  │Data Contracts│  │  Retention    │            │
│  │(dashboard)│ │ (SLAs)       │  │ (lifecycle)   │            │
│  └──────────┘  └──────────────┘  └───────────────┘            │
└─────────────────────────────────────────────────────────────────┘
```

**Time estimate:** 6-8 hours | **Difficulty:** Advanced

**Prerequisites:** Notebooks 01-30 (techmart_dw database)

---

---
# 📗 Section 1: Discovery & Cataloging

## Auto-Discovery

A data catalog answers: *What data do we have, where is it, and what does it look like?*

| Catalog Attribute | Description |
|---|---|
| Database | Which database/schema |
| Table | Table name |
| Columns | Column names and types |
| Row Count | How much data |
| Freshness | When was it last updated |
| Owner | Who is responsible |
| Classification | PII/PHI/Financial/Public |

In [0]:
# Auto-discover all tables across all databases
from datetime import datetime
import re

class DataCatalog:
    """Auto-discovers and catalogs all tables in the environment."""
    
    def __init__(self):
        self.entries = []
    
    def discover_database(self, database):
        """Scan a database and catalog all tables."""
        try:
            tables = spark.sql(f"SHOW TABLES IN {database}").collect()
        except Exception as e:
            print(f"  ⚠️ Cannot access {database}: {e}")
            return
        
        for table_row in tables:
            table_name = table_row.tableName
            full_name = f"{database}.{table_name}"
            
            try:
                df = spark.table(full_name)
                schema_info = [
                    {"name": f.name, "type": str(f.dataType), "nullable": f.nullable}
                    for f in df.schema.fields
                ]
                row_count = df.count()
                
                self.entries.append({
                    "database": database,
                    "table_name": table_name,
                    "full_name": full_name,
                    "columns": schema_info,
                    "column_count": len(schema_info),
                    "row_count": row_count,
                    "discovered_at": datetime.now().isoformat(),
                    "owner": "unassigned",
                    "description": "",
                    "classification": "unclassified"
                })
            except Exception as e:
                self.entries.append({
                    "database": database,
                    "table_name": table_name,
                    "full_name": full_name,
                    "columns": [],
                    "column_count": 0,
                    "row_count": -1,
                    "discovered_at": datetime.now().isoformat(),
                    "owner": "unassigned",
                    "description": "",
                    "classification": "error",
                    "error": str(e)
                })
    
    def discover_all(self, databases):
        """Discover all tables across multiple databases."""
        print(f"🔍 Discovering tables across {len(databases)} database(s)...")
        for db in databases:
            print(f"  Scanning {db}...")
            self.discover_database(db)
        print(f"  ✅ Found {len(self.entries)} tables total")
    
    def summary(self):
        """Print catalog summary."""
        print(f"
{'='*70}")
        print(f"  📚 DATA CATALOG SUMMARY")
        print(f"{'='*70}")
        print(f"  {'Database':<20} {'Table':<30} {'Columns':<10} {'Rows':<12}")
        print(f"  {'-'*70}")
        
        total_rows = 0
        for entry in sorted(self.entries, key=lambda x: x['full_name']):
            rows = f"{entry['row_count']:,}" if entry['row_count'] >= 0 else "ERROR"
            print(f"  {entry['database']:<20} {entry['table_name']:<30} {entry['column_count']:<10} {rows:<12}")
            if entry['row_count'] > 0:
                total_rows += entry['row_count']
        
        print(f"  {'-'*70}")
        print(f"  Total: {len(self.entries)} tables, {total_rows:,} rows")
        print(f"{'='*70}")
    
    def to_dataframe(self):
        """Convert catalog to a Spark DataFrame for persistence."""
        from pyspark.sql import Row
        rows = []
        for entry in self.entries:
            col_names = [c['name'] for c in entry['columns']]
            rows.append(Row(
                database=entry['database'],
                table_name=entry['table_name'],
                full_name=entry['full_name'],
                column_count=entry['column_count'],
                row_count=entry['row_count'],
                column_names=", ".join(col_names),
                discovered_at=entry['discovered_at'],
                owner=entry['owner'],
                classification=entry['classification']
            ))
        return spark.createDataFrame(rows) if rows else None


# Discover techmart_dw
catalog = DataCatalog()
catalog.discover_all(["techmart_dw"])
catalog.summary()


In [0]:
# Persist catalog to a Delta table
catalog_df = catalog.to_dataframe()
if catalog_df:
    catalog_df.write.mode("overwrite").saveAsTable("techmart_dw.governance_catalog")
    print("✅ Catalog persisted to techmart_dw.governance_catalog")
    spark.sql("SELECT database, table_name, column_count, row_count FROM techmart_dw.governance_catalog ORDER BY row_count DESC").show(truncate=False)


In [0]:
# ============================================
# 🎯 YOUR TURN: Enrich the Catalog
# ============================================
# Add metadata enrichment to the catalog:
# 1. Add an "owner" for each table (assign based on table name patterns):
#    - Tables with 'customer' → "customer_team"
#    - Tables with 'order' → "commerce_team"
#    - Tables with 'product' → "catalog_team"
#    - Everything else → "platform_team"
# 2. Add a "description" for at least 5 tables
# 3. Update the governance_catalog table with this enrichment
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
spark.sql("""
UPDATE techmart_dw.governance_catalog
SET owner = CASE
    WHEN table_name LIKE '%customer%' THEN 'customer_team'
    WHEN table_name LIKE '%order%' THEN 'commerce_team'
    WHEN table_name LIKE '%product%' THEN 'catalog_team'
    WHEN table_name LIKE '%payment%' THEN 'finance_team'
    WHEN table_name LIKE '%ship%' THEN 'logistics_team'
    ELSE 'platform_team'
END
""")

# Add descriptions for key tables
descriptions = {
    "customers": "Master customer records with demographics and lifetime value",
    "orders": "Transaction records for all customer orders",
    "products": "Product catalog with pricing and categories",
    "payments": "Payment transactions linked to orders",
    "order_items": "Line-item details for each order"
}

for table, desc in descriptions.items():
    spark.sql(f"UPDATE techmart_dw.governance_catalog SET classification = '{desc}' WHERE table_name = '{table}'")

print("✅ Catalog enriched with owners and descriptions")
spark.sql("SELECT table_name, owner, classification FROM techmart_dw.governance_catalog WHERE owner != 'platform_team'").show(truncate=False)

---
# 📗 Section 2: Data Classification

## Classification Categories

| Category | Description | Examples | Sensitivity |
|---|---|---|---|
| **PII** | Personally Identifiable Information | email, phone, name, address | High |
| **PHI** | Protected Health Information | diagnosis, patient_id, medical_code | Critical |
| **Financial** | Financial data | account_number, amount, credit_score | High |
| **Internal** | Business-sensitive | revenue, cost, margin | Medium |
| **Public** | Non-sensitive | product_name, category | Low |

In [0]:
# Data Classification Engine
import re

class DataClassifier:
    """Classifies columns based on name patterns and content analysis."""
    
    # PII patterns (column name matching)
    PII_PATTERNS = {
        "email": [r"email", r"e_mail", r"mail_address"],
        "phone": [r"phone", r"mobile", r"cell", r"tel"],
        "name": [r"first_name", r"last_name", r"full_name", r"customer_name"],
        "address": [r"address", r"street", r"city", r"zip", r"postal"],
        "ssn": [r"ssn", r"social_security", r"national_id"],
        "ip_address": [r"ip_addr", r"ip_address", r"client_ip"],
        "dob": [r"date_of_birth", r"dob", r"birth_date"]
    }
    
    # Financial patterns
    FINANCIAL_PATTERNS = {
        "account": [r"account_num", r"account_id", r"bank_account"],
        "amount": [r"amount", r"price", r"cost", r"revenue", r"salary", r"income"],
        "credit": [r"credit_score", r"credit_limit", r"credit_card"],
        "transaction": [r"transaction_id", r"payment_id"]
    }
    
    # Internal business patterns
    INTERNAL_PATTERNS = {
        "margin": [r"margin", r"profit", r"markup"],
        "cost": [r"unit_cost", r"cogs", r"operating_cost"],
        "strategy": [r"segment", r"tier", r"score", r"lifetime_value"]
    }
    
    def __init__(self):
        self.classifications = []
    
    def classify_column(self, table_name, column_name, data_type):
        """Classify a single column."""
        col_lower = column_name.lower()
        
        # Check PII
        for pii_type, patterns in self.PII_PATTERNS.items():
            for pattern in patterns:
                if re.search(pattern, col_lower):
                    return {
                        "table": table_name,
                        "column": column_name,
                        "data_type": data_type,
                        "classification": "PII",
                        "sub_type": pii_type,
                        "sensitivity": "high",
                        "requires_masking": True
                    }
        
        # Check Financial
        for fin_type, patterns in self.FINANCIAL_PATTERNS.items():
            for pattern in patterns:
                if re.search(pattern, col_lower):
                    return {
                        "table": table_name,
                        "column": column_name,
                        "data_type": data_type,
                        "classification": "Financial",
                        "sub_type": fin_type,
                        "sensitivity": "high",
                        "requires_masking": False
                    }
        
        # Check Internal
        for int_type, patterns in self.INTERNAL_PATTERNS.items():
            for pattern in patterns:
                if re.search(pattern, col_lower):
                    return {
                        "table": table_name,
                        "column": column_name,
                        "data_type": data_type,
                        "classification": "Internal",
                        "sub_type": int_type,
                        "sensitivity": "medium",
                        "requires_masking": False
                    }
        
        # Default: Public
        return {
            "table": table_name,
            "column": column_name,
            "data_type": data_type,
            "classification": "Public",
            "sub_type": "general",
            "sensitivity": "low",
            "requires_masking": False
        }
    
    def classify_table(self, table_name):
        """Classify all columns in a table."""
        df = spark.table(table_name)
        results = []
        for field in df.schema.fields:
            result = self.classify_column(table_name, field.name, str(field.dataType))
            results.append(result)
            self.classifications.append(result)
        return results
    
    def classify_database(self, database):
        """Classify all tables in a database."""
        tables = spark.sql(f"SHOW TABLES IN {database}").collect()
        print(f"🏷️ Classifying {len(tables)} tables in {database}...")
        for table_row in tables:
            try:
                self.classify_table(f"{database}.{table_row.tableName}")
            except:
                pass
        print(f"  ✅ Classified {len(self.classifications)} columns")
    
    def report(self):
        """Generate classification report."""
        print(f"
{'='*70}")
        print(f"  🏷️ DATA CLASSIFICATION REPORT")
        print(f"{'='*70}")
        
        # Summary by classification
        from collections import Counter
        class_counts = Counter(c['classification'] for c in self.classifications)
        print(f"
  Summary:")
        for cls, count in class_counts.most_common():
            pct = count / len(self.classifications) * 100
            icon = {"PII": "🔴", "Financial": "🟠", "Internal": "🟡", "Public": "🟢"}.get(cls, "⚪")
            print(f"    {icon} {cls}: {count} columns ({pct:.1f}%)")
        
        # PII details
        pii_cols = [c for c in self.classifications if c['classification'] == 'PII']
        if pii_cols:
            print(f"
  🔴 PII Columns Requiring Protection:")
            for col in pii_cols:
                print(f"    • {col['table']}.{col['column']} ({col['sub_type']})")
        
        # Financial details
        fin_cols = [c for c in self.classifications if c['classification'] == 'Financial']
        if fin_cols:
            print(f"
  🟠 Financial Columns:")
            for col in fin_cols[:10]:
                print(f"    • {col['table']}.{col['column']} ({col['sub_type']})")
            if len(fin_cols) > 10:
                print(f"    ... and {len(fin_cols) - 10} more")
        
        print(f"
{'='*70}")


# Run classification
classifier = DataClassifier()
classifier.classify_database("techmart_dw")
classifier.report()


In [0]:
# Persist classification results
from pyspark.sql import Row

class_rows = [Row(**c) for c in classifier.classifications]
class_df = spark.createDataFrame(class_rows)
class_df.write.mode("overwrite").saveAsTable("techmart_dw.governance_classifications")

print("✅ Classifications persisted to techmart_dw.governance_classifications")
spark.sql("""
    SELECT classification, sub_type, COUNT(*) AS column_count,
           COLLECT_SET(CONCAT(table, '.', column)) AS sample_columns
    FROM techmart_dw.governance_classifications
    WHERE classification != 'Public'
    GROUP BY classification, sub_type
    ORDER BY classification, column_count DESC
""").show(truncate=60)


In [0]:
# ============================================
# 🎯 YOUR TURN: Add Custom Classification
# ============================================
# Extend the DataClassifier to detect:
# 1. "Temporal" data — columns with date/time patterns
#    (patterns: date, time, timestamp, created_at, updated_at, _at)
# 2. "Identifier" data — primary/foreign keys
#    (patterns: _id, _key, _sk, _pk)
#
# Re-classify the database and show how many columns fall into each new category.
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
class ExtendedClassifier(DataClassifier):
    """Extended classifier with temporal and identifier detection."""
    
    TEMPORAL_PATTERNS = [r"date", r"time", r"timestamp", r"_at$", r"created", r"updated", r"expired"]
    IDENTIFIER_PATTERNS = [r"_id$", r"_key$", r"_sk$", r"_pk$", r"^id$"]
    
    def classify_column(self, table_name, column_name, data_type):
        """Override to add temporal and identifier detection."""
        # First check parent classifications
        result = super().classify_column(table_name, column_name, data_type)
        
        # If already classified as PII/Financial, keep that
        if result['classification'] in ('PII', 'Financial'):
            return result
        
        col_lower = column_name.lower()
        
        # Check Temporal
        for pattern in self.TEMPORAL_PATTERNS:
            if re.search(pattern, col_lower):
                return {
                    "table": table_name,
                    "column": column_name,
                    "data_type": data_type,
                    "classification": "Temporal",
                    "sub_type": "datetime",
                    "sensitivity": "low",
                    "requires_masking": False
                }
        
        # Check Identifier
        for pattern in self.IDENTIFIER_PATTERNS:
            if re.search(pattern, col_lower):
                return {
                    "table": table_name,
                    "column": column_name,
                    "data_type": data_type,
                    "classification": "Identifier",
                    "sub_type": "key",
                    "sensitivity": "low",
                    "requires_masking": False
                }
        
        return result

# Re-classify with extended classifier
ext_classifier = ExtendedClassifier()
ext_classifier.classify_database("techmart_dw")
ext_classifier.report()


---
# 📗 Section 3: Quality Rules Engine

## Quality Dimensions

| Dimension | What it Measures | Example Check |
|---|---|---|
| **Completeness** | No missing values | NULL rate < 1% |
| **Uniqueness** | No duplicates | PK is unique |
| **Validity** | Values in expected range | Status IN ('active', 'inactive') |
| **Freshness** | Data is recent | Last update < 24 hours |
| **Consistency** | Cross-table agreement | FK exists in parent |
| **Volume** | Expected row count | Between 1000 and 1M rows |

In [0]:
# Quality Rules Engine
from datetime import datetime

class QualityRule:
    """A single quality rule definition."""
    def __init__(self, name, table, dimension, check_sql, threshold=0, severity="error"):
        self.name = name
        self.table = table
        self.dimension = dimension
        self.check_sql = check_sql
        self.threshold = threshold  # max allowed failures
        self.severity = severity


class QualityEngine:
    """Evaluates quality rules and produces scores."""
    
    def __init__(self):
        self.rules = []
        self.results = []
    
    def add_rule(self, name, table, dimension, check_sql, threshold=0, severity="error"):
        """Register a quality rule."""
        self.rules.append(QualityRule(name, table, dimension, check_sql, threshold, severity))
    
    def add_completeness_check(self, table, column, max_null_pct=1.0):
        """Add a completeness check for a column."""
        sql = f"""
            SELECT COUNT(*) FROM {table} 
            WHERE {column} IS NULL
        """
        self.add_rule(
            f"completeness_{table.split('.')[-1]}_{column}",
            table, "completeness", sql,
            threshold=int(max_null_pct * spark.table(table).count() / 100),
            severity="error" if max_null_pct == 0 else "warning"
        )
    
    def add_uniqueness_check(self, table, column):
        """Add a uniqueness check for a column."""
        sql = f"""
            SELECT COUNT(*) FROM (
                SELECT {column} FROM {table}
                GROUP BY {column} HAVING COUNT(*) > 1
            )
        """
        self.add_rule(f"unique_{table.split('.')[-1]}_{column}", table, "uniqueness", sql)
    
    def add_validity_check(self, table, column, valid_values):
        """Add a validity check for allowed values."""
        values_str = ", ".join([f"'{v}'" for v in valid_values])
        sql = f"""
            SELECT COUNT(*) FROM {table}
            WHERE {column} NOT IN ({values_str}) AND {column} IS NOT NULL
        """
        self.add_rule(f"valid_{table.split('.')[-1]}_{column}", table, "validity", sql)
    
    def add_volume_check(self, table, min_rows, max_rows):
        """Add a volume check."""
        sql = f"""
            SELECT CASE 
                WHEN (SELECT COUNT(*) FROM {table}) < {min_rows} THEN 1
                WHEN (SELECT COUNT(*) FROM {table}) > {max_rows} THEN 1
                ELSE 0
            END
        """
        self.add_rule(f"volume_{table.split('.')[-1]}", table, "volume", sql)
    
    def run_all(self):
        """Execute all quality rules."""
        print(f"🧪 Running {len(self.rules)} quality rules...")
        self.results = []
        
        for rule in self.rules:
            try:
                failures = spark.sql(rule.check_sql).collect()[0][0]
                passed = failures <= rule.threshold
                self.results.append({
                    "rule": rule.name,
                    "table": rule.table,
                    "dimension": rule.dimension,
                    "passed": passed,
                    "failures": failures,
                    "threshold": rule.threshold,
                    "severity": rule.severity,
                    "checked_at": datetime.now().isoformat()
                })
            except Exception as e:
                self.results.append({
                    "rule": rule.name,
                    "table": rule.table,
                    "dimension": rule.dimension,
                    "passed": False,
                    "failures": -1,
                    "threshold": rule.threshold,
                    "severity": "error",
                    "checked_at": datetime.now().isoformat(),
                    "error": str(e)
                })
        
        passed = sum(1 for r in self.results if r['passed'])
        failed = len(self.results) - passed
        print(f"  ✅ Passed: {passed} | ❌ Failed: {failed}")
    
    def score(self, table=None):
        """Calculate quality score (0-100)."""
        results = self.results
        if table:
            results = [r for r in results if r['table'] == table]
        if not results:
            return 0
        passed = sum(1 for r in results if r['passed'])
        return round(passed / len(results) * 100, 1)
    
    def dashboard(self):
        """Print quality dashboard."""
        print(f"
{'='*70}")
        print(f"  📊 DATA QUALITY DASHBOARD")
        print(f"  Overall Score: {self.score()}%")
        print(f"{'='*70}")
        
        # Score by table
        tables = set(r['table'] for r in self.results)
        print(f"
  Score by Table:")
        for table in sorted(tables):
            score = self.score(table)
            bar = "█" * int(score / 5) + "░" * (20 - int(score / 5))
            icon = "✅" if score >= 90 else "⚠️" if score >= 70 else "❌"
            print(f"    {icon} {table.split('.')[-1]:<25} {bar} {score}%")
        
        # Score by dimension
        print(f"
  Score by Dimension:")
        dimensions = set(r['dimension'] for r in self.results)
        for dim in sorted(dimensions):
            dim_results = [r for r in self.results if r['dimension'] == dim]
            passed = sum(1 for r in dim_results if r['passed'])
            score = round(passed / len(dim_results) * 100, 1)
            print(f"    {dim:<15}: {score}% ({passed}/{len(dim_results)} passed)")
        
        # Failed rules
        failed = [r for r in self.results if not r['passed']]
        if failed:
            print(f"
  ❌ Failed Rules ({len(failed)}):")
            for r in failed[:10]:
                print(f"    • {r['rule']}: {r['failures']} failures (threshold: {r['threshold']})")
        
        print(f"
{'='*70}")


# Configure quality rules for techmart_dw
qe = QualityEngine()

# Completeness checks
qe.add_completeness_check("techmart_dw.customers", "customer_id", max_null_pct=0)
qe.add_completeness_check("techmart_dw.customers", "email", max_null_pct=1)
qe.add_completeness_check("techmart_dw.orders", "order_id", max_null_pct=0)
qe.add_completeness_check("techmart_dw.orders", "customer_id", max_null_pct=0)
qe.add_completeness_check("techmart_dw.orders", "total_amount", max_null_pct=0)

# Uniqueness checks
qe.add_uniqueness_check("techmart_dw.customers", "customer_id")
qe.add_uniqueness_check("techmart_dw.orders", "order_id")
qe.add_uniqueness_check("techmart_dw.products", "product_id")

# Validity checks
qe.add_validity_check("techmart_dw.orders", "status",
    ["completed", "shipped", "processing", "pending", "cancelled", "returned"])
qe.add_validity_check("techmart_dw.customers", "customer_segment",
    ["Premium", "Standard", "Basic", "Enterprise"])

# Volume checks
qe.add_volume_check("techmart_dw.customers", 1000, 100000)
qe.add_volume_check("techmart_dw.orders", 5000, 500000)
qe.add_volume_check("techmart_dw.products", 100, 10000)

# Run and display
qe.run_all()
qe.dashboard()


In [0]:
# Persist quality results
from pyspark.sql import Row

quality_rows = [Row(**{k: v for k, v in r.items() if k != 'error'}) for r in qe.results]
quality_df = spark.createDataFrame(quality_rows)
quality_df.write.mode("overwrite").saveAsTable("techmart_dw.governance_quality_results")
print("✅ Quality results persisted to techmart_dw.governance_quality_results")


In [0]:
# ============================================
# 🎯 YOUR TURN: Add Quality Rules
# ============================================
# Add quality rules for the payments table:
# 1. Completeness: payment_id NOT NULL (0% threshold)
# 2. Completeness: amount NOT NULL (0% threshold)
# 3. Uniqueness: payment_id is unique
# 4. Validity: payment_status IN ('completed', 'pending', 'failed', 'refunded')
# 5. Custom: amount > 0 (no negative payments)
#    SQL: SELECT COUNT(*) FROM techmart_dw.payments WHERE amount <= 0
# 6. Volume: between 10000 and 100000 rows
#
# Run and show the updated dashboard.
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
qe2 = QualityEngine()

# Payment rules
qe2.add_completeness_check("techmart_dw.payments", "payment_id", max_null_pct=0)
qe2.add_completeness_check("techmart_dw.payments", "amount", max_null_pct=0)
qe2.add_uniqueness_check("techmart_dw.payments", "payment_id")
qe2.add_validity_check("techmart_dw.payments", "payment_status",
    ["completed", "pending", "failed", "refunded"])
qe2.add_rule("positive_amount_payments", "techmart_dw.payments", "validity",
    "SELECT COUNT(*) FROM techmart_dw.payments WHERE amount <= 0")
qe2.add_volume_check("techmart_dw.payments", 10000, 100000)

qe2.run_all()
qe2.dashboard()


---
# 📗 Section 4: Access Control Matrix

## Role-Based Access Control (RBAC)

| Role | Description | Access Level |
|---|---|---|
| **analyst** | Business analysts, dashboard users | Read public + internal |
| **engineer** | Data engineers, pipeline developers | Read/write all non-PII |
| **data_scientist** | ML engineers | Read all (masked PII) |
| **admin** | Platform administrators | Full access |
| **compliance** | Compliance officers | Read all + audit logs |

In [0]:
# Access Control Matrix
class AccessControlMatrix:
    """Defines and manages role-based access control."""
    
    def __init__(self):
        self.roles = {}
        self.table_policies = {}
    
    def define_role(self, role_name, description, default_access="none"):
        """Define a role."""
        self.roles[role_name] = {
            "description": description,
            "default_access": default_access,
            "table_overrides": {}
        }
    
    def set_table_policy(self, table_name, classification, pii_columns=None):
        """Set access policy for a table based on classification."""
        self.table_policies[table_name] = {
            "classification": classification,
            "pii_columns": pii_columns or []
        }
    
    def grant(self, role_name, table_name, access_level):
        """Grant access to a role for a table."""
        if role_name in self.roles:
            self.roles[role_name]["table_overrides"][table_name] = access_level
    
    def check_access(self, role_name, table_name, column_name=None):
        """Check if a role has access to a table/column."""
        if role_name not in self.roles:
            return "denied"
        
        role = self.roles[role_name]
        
        # Check table-level override
        if table_name in role["table_overrides"]:
            access = role["table_overrides"][table_name]
        else:
            access = role["default_access"]
        
        # Check column-level PII restriction
        if column_name and table_name in self.table_policies:
            policy = self.table_policies[table_name]
            if column_name in policy["pii_columns"]:
                if role_name not in ("admin", "compliance"):
                    return "masked"
        
        return access
    
    def generate_grants(self, role_name):
        """Generate SQL GRANT statements for a role."""
        statements = []
        role = self.roles[role_name]
        
        for table, access in role["table_overrides"].items():
            if access == "read":
                statements.append(f"GRANT SELECT ON TABLE {table} TO `{role_name}`;")
            elif access == "write":
                statements.append(f"GRANT SELECT, MODIFY ON TABLE {table} TO `{role_name}`;")
            elif access == "admin":
                statements.append(f"GRANT ALL PRIVILEGES ON TABLE {table} TO `{role_name}`;")
        
        return statements
    
    def matrix_view(self):
        """Print the access control matrix."""
        print(f"
{'='*80}")
        print(f"  🔐 ACCESS CONTROL MATRIX")
        print(f"{'='*80}")
        
        tables = sorted(set(
            t for role in self.roles.values() 
            for t in role["table_overrides"].keys()
        ))
        
        # Header
        role_names = list(self.roles.keys())
        header = f"  {'Table':<30}" + "".join(f"{r:<12}" for r in role_names)
        print(header)
        print(f"  {'-'*78}")
        
        for table in tables:
            short_name = table.split(".")[-1][:28]
            row = f"  {short_name:<30}"
            for role_name in role_names:
                access = self.check_access(role_name, table)
                icon = {"read": "👁️", "write": "✏️", "admin": "🔑", "none": "🚫", "masked": "🔒"}
                row += f"{icon.get(access, '?'):<12}"
            print(row)
        
        print(f"
  Legend: 👁️=Read  ✏️=Write  🔑=Admin  🚫=None  🔒=Masked")
        print(f"{'='*80}")


# Define roles
acm = AccessControlMatrix()
acm.define_role("analyst", "Business analysts", default_access="none")
acm.define_role("engineer", "Data engineers", default_access="read")
acm.define_role("data_scientist", "ML engineers", default_access="read")
acm.define_role("admin", "Platform admins", default_access="admin")
acm.define_role("compliance", "Compliance officers", default_access="read")

# Set table policies
acm.set_table_policy("techmart_dw.customers", "PII", 
    pii_columns=["email", "first_name", "last_name", "phone"])
acm.set_table_policy("techmart_dw.orders", "Internal")
acm.set_table_policy("techmart_dw.products", "Public")
acm.set_table_policy("techmart_dw.payments", "Financial")

# Grant access
tables = ["techmart_dw.customers", "techmart_dw.orders", 
          "techmart_dw.products", "techmart_dw.payments"]

for table in tables:
    acm.grant("analyst", table, "read")
    acm.grant("engineer", table, "write")
    acm.grant("data_scientist", table, "read")
    acm.grant("admin", table, "admin")
    acm.grant("compliance", table, "read")

acm.matrix_view()

# Generate GRANT statements
print("
📝 SQL GRANT statements for 'analyst' role:")
for stmt in acm.generate_grants("analyst"):
    print(f"  {stmt}")


---
# 📗 Section 5: Lineage Tracking

## Data Lineage = Knowing Where Data Comes From

Lineage answers:
- Where did this data originate?
- What transformations were applied?
- If I change table X, what downstream tables break?

In [0]:
# Lineage Tracking System
class LineageTracker:
    """Tracks data lineage (source → transform → target)."""
    
    def __init__(self):
        self.edges = []  # (source, target, transform_description)
    
    def add_edge(self, source, target, transform=""):
        """Add a lineage edge."""
        self.edges.append({
            "source": source,
            "target": target,
            "transform": transform
        })
    
    def get_upstream(self, table):
        """Get all upstream dependencies (what feeds this table)."""
        upstream = set()
        to_check = [table]
        while to_check:
            current = to_check.pop()
            for edge in self.edges:
                if edge["target"] == current and edge["source"] not in upstream:
                    upstream.add(edge["source"])
                    to_check.append(edge["source"])
        return upstream
    
    def get_downstream(self, table):
        """Get all downstream dependents (what this table feeds)."""
        downstream = set()
        to_check = [table]
        while to_check:
            current = to_check.pop()
            for edge in self.edges:
                if edge["source"] == current and edge["target"] not in downstream:
                    downstream.add(edge["target"])
                    to_check.append(edge["target"])
        return downstream
    
    def impact_analysis(self, table):
        """Analyze impact of changing a table."""
        downstream = self.get_downstream(table)
        print(f"
⚠️ IMPACT ANALYSIS: Changes to '{table}'")
        print(f"   Would affect {len(downstream)} downstream table(s):")
        for d in sorted(downstream):
            print(f"     → {d}")
        return downstream
    
    def print_dag(self):
        """Print the lineage DAG."""
        print(f"
{'='*60}")
        print(f"  📊 DATA LINEAGE DAG")
        print(f"{'='*60}")
        
        # Group by layer
        sources = set()
        targets = set()
        for edge in self.edges:
            sources.add(edge["source"])
            targets.add(edge["target"])
        
        roots = sources - targets  # tables that are only sources
        leaves = targets - sources  # tables that are only targets
        middle = sources & targets  # tables that are both
        
        print(f"
  🟢 Sources (raw): {sorted(roots)}")
        print(f"  🟡 Intermediate: {sorted(middle)}")
        print(f"  🔵 Final (Gold): {sorted(leaves)}")
        
        print(f"
  Edges:")
        for edge in self.edges:
            transform = f" [{edge['transform']}]" if edge['transform'] else ""
            print(f"    {edge['source']} → {edge['target']}{transform}")
        print(f"{'='*60}")


# Define lineage for TechMart
lineage = LineageTracker()

# Raw → Bronze (ingestion)
lineage.add_edge("external.orders_api", "techmart_dw.orders", "API ingestion")
lineage.add_edge("external.customer_db", "techmart_dw.customers", "CDC replication")
lineage.add_edge("external.product_catalog", "techmart_dw.products", "Full sync")
lineage.add_edge("external.payment_gateway", "techmart_dw.payments", "Event stream")

# Bronze → Silver (cleaning)
lineage.add_edge("techmart_dw.orders", "techmart_dw.stg_orders", "Clean + standardize")
lineage.add_edge("techmart_dw.customers", "techmart_dw.stg_customers", "Clean + dedupe")
lineage.add_edge("techmart_dw.products", "techmart_dw.stg_products", "Clean + enrich")

# Silver → Gold (aggregation)
lineage.add_edge("techmart_dw.stg_orders", "techmart_dw.gold_daily_sales", "Aggregate by day")
lineage.add_edge("techmart_dw.stg_customers", "techmart_dw.gold_customer_360", "Join + score")
lineage.add_edge("techmart_dw.stg_orders", "techmart_dw.gold_customer_360", "Order history")

lineage.print_dag()

# Impact analysis
lineage.impact_analysis("techmart_dw.orders")
lineage.impact_analysis("techmart_dw.stg_orders")


---
# 📗 Section 6: PII Masking Implementation

## Masking Strategies

| Strategy | Method | Example | Use Case |
|---|---|---|---|
| **Hash** | SHA-256 one-way hash | `john@email.com` → `a3f2b...` | Analytics (join-safe) |
| **Redact** | Replace with fixed string | `john@email.com` → `***REDACTED***` | Display |
| **Generalize** | Reduce precision | `john@email.com` → `***@email.com` | Partial visibility |
| **Tokenize** | Replace with random token | `john@email.com` → `TOK_8f3a2b` | Reversible masking |

In [0]:
# PII Masking Functions
from pyspark.sql.functions import sha2, lit, concat, substring, regexp_replace, col, when

# Create masked view of customers table
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.v_customers_masked AS
SELECT
    customer_id,
    -- Hash email (preserves join capability)
    SHA2(email, 256) AS email_hash,
    -- Redact first/last name
    CONCAT(SUBSTRING(first_name, 1, 1), '***') AS first_name,
    CONCAT(SUBSTRING(last_name, 1, 1), '***') AS last_name,
    -- Keep non-PII columns as-is
    customer_segment,
    lifetime_value,
    signup_date,
    is_active
FROM techmart_dw.customers
""")

print("✅ Created masked view: v_customers_masked")
print("\n📊 Original vs Masked:")
print("\nOriginal (first 3 rows):")
spark.sql("SELECT customer_id, email, first_name, last_name FROM techmart_dw.customers LIMIT 3").show(truncate=False)
print("Masked (first 3 rows):")
spark.sql("SELECT customer_id, email_hash, first_name, last_name FROM techmart_dw.v_customers_masked LIMIT 3").show(truncate=False)

In [0]:
# ============================================
# 🎯 YOUR TURN: Create Masked Views
# ============================================
# Create a masked view for the payments table:
# 1. Keep: payment_id, order_id, payment_status, payment_date
# 2. Mask amount: show only the magnitude (e.g., $1,234.56 → "$1,000-$2,000 range")
# 3. Mask payment_method: show only first 4 chars + "****"
# 4. Name it: techmart_dw.v_payments_masked
#
# Show original vs masked for comparison.
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.v_payments_masked AS
SELECT
    payment_id,
    order_id,
    CONCAT(SUBSTRING(payment_method, 1, 4), '****') AS payment_method_masked,
    CASE
        WHEN amount < 100 THEN '$0-$100'
        WHEN amount < 500 THEN '$100-$500'
        WHEN amount < 1000 THEN '$500-$1,000'
        WHEN amount < 5000 THEN '$1,000-$5,000'
        ELSE '$5,000+'
    END AS amount_range,
    payment_status,
    payment_date
FROM techmart_dw.payments
""")

print("✅ Created masked view: v_payments_masked")
print("\nOriginal:")
spark.sql("SELECT payment_id, payment_method, amount, payment_status FROM techmart_dw.payments LIMIT 5").show()
print("Masked:")
spark.sql("SELECT * FROM techmart_dw.v_payments_masked LIMIT 5").show()

---
# 📗 Section 7: Data Contracts

## What is a Data Contract?

A formal agreement between a data **producer** and **consumer** that defines:
- **Schema** — expected columns, types, nullability
- **Quality SLAs** — completeness, freshness, accuracy
- **Volume** — expected row count range
- **Semantics** — what each column means

In [0]:
# Data Contract System
class DataContract:
    """Defines a contract between data producer and consumer."""
    
    def __init__(self, name, producer, consumer, table):
        self.name = name
        self.producer = producer
        self.consumer = consumer
        self.table = table
        self.schema_contract = {}
        self.quality_slas = {}
        self.volume_range = (0, float('inf'))
    
    def expect_columns(self, columns):
        """Define expected columns with types."""
        self.schema_contract = columns
    
    def expect_quality(self, **slas):
        """Define quality SLAs."""
        self.quality_slas = slas
    
    def expect_volume(self, min_rows, max_rows):
        """Define expected volume range."""
        self.volume_range = (min_rows, max_rows)
    
    def validate(self):
        """Validate the contract against actual data."""
        print(f"
📋 Validating Contract: {self.name}")
        print(f"   Producer: {self.producer} | Consumer: {self.consumer}")
        print(f"   Table: {self.table}")
        print(f"   {'─'*50}")
        
        violations = []
        
        # Schema validation
        if self.schema_contract:
            df = spark.table(self.table)
            actual_cols = {f.name: str(f.dataType) for f in df.schema.fields}
            
            for col_name, expected_type in self.schema_contract.items():
                if col_name not in actual_cols:
                    violations.append(f"Missing column: {col_name}")
                    print(f"   ❌ Schema: Missing column '{col_name}'")
                else:
                    print(f"   ✅ Schema: '{col_name}' exists")
        
        # Volume validation
        row_count = spark.table(self.table).count()
        min_r, max_r = self.volume_range
        if min_r <= row_count <= max_r:
            print(f"   ✅ Volume: {row_count:,} rows (expected {min_r:,}-{max_r:,})")
        else:
            violations.append(f"Volume out of range: {row_count} (expected {min_r}-{max_r})")
            print(f"   ❌ Volume: {row_count:,} rows (expected {min_r:,}-{max_r:,})")
        
        # Quality SLA validation
        if "max_null_pct" in self.quality_slas:
            for col_name in self.schema_contract:
                try:
                    null_count = spark.sql(f"SELECT COUNT(*) FROM {self.table} WHERE {col_name} IS NULL").collect()[0][0]
                    null_pct = null_count / row_count * 100
                    threshold = self.quality_slas["max_null_pct"]
                    if null_pct <= threshold:
                        print(f"   ✅ Quality: '{col_name}' NULL rate {null_pct:.2f}% ≤ {threshold}%")
                    else:
                        violations.append(f"{col_name} NULL rate {null_pct:.2f}% > {threshold}%")
                        print(f"   ❌ Quality: '{col_name}' NULL rate {null_pct:.2f}% > {threshold}%")
                except:
                    pass
        
        # Summary
        if violations:
            print(f"
   🚨 CONTRACT VIOLATED: {len(violations)} issue(s)")
        else:
            print(f"
   ✅ CONTRACT SATISFIED")
        
        return len(violations) == 0


# Define contracts
orders_contract = DataContract(
    name="Orders Data Contract",
    producer="commerce_team",
    consumer="analytics_team",
    table="techmart_dw.orders"
)
orders_contract.expect_columns({
    "order_id": "IntegerType()",
    "customer_id": "IntegerType()",
    "total_amount": "DecimalType()",
    "status": "StringType()",
    "order_date": "DateType()"
})
orders_contract.expect_volume(10000, 500000)
orders_contract.expect_quality(max_null_pct=1.0)

customers_contract = DataContract(
    name="Customers Data Contract",
    producer="customer_team",
    consumer="analytics_team",
    table="techmart_dw.customers"
)
customers_contract.expect_columns({
    "customer_id": "IntegerType()",
    "email": "StringType()",
    "customer_segment": "StringType()",
    "lifetime_value": "DecimalType()"
})
customers_contract.expect_volume(1000, 100000)
customers_contract.expect_quality(max_null_pct=2.0)

# Validate contracts
orders_contract.validate()
customers_contract.validate()


In [0]:
# ============================================
# 🎯 YOUR TURN: Define a Data Contract
# ============================================
# Create a DataContract for the products table:
# - Producer: "catalog_team"
# - Consumer: "recommendation_engine"
# - Expected columns: product_id, product_name, category, price, cost
# - Volume: 100 to 10000 rows
# - Quality: max 0.5% NULL rate
#
# Validate the contract.
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
products_contract = DataContract(
    name="Products Data Contract",
    producer="catalog_team",
    consumer="recommendation_engine",
    table="techmart_dw.products"
)
products_contract.expect_columns({
    "product_id": "IntegerType()",
    "product_name": "StringType()",
    "category": "StringType()",
    "price": "DecimalType()",
    "cost": "DecimalType()"
})
products_contract.expect_volume(100, 10000)
products_contract.expect_quality(max_null_pct=0.5)

products_contract.validate()


---
# 📗 Section 8: Monitoring & Compliance Assessment

## Governance Monitoring
- Track quality scores over time
- Detect classification drift (new PII appearing)
- Monitor access patterns
- Generate compliance reports

In [0]:
# Compliance Assessment
class ComplianceAssessor:
    """Assess GDPR and data governance compliance."""
    
    def __init__(self, catalog, classifier, quality_engine):
        self.catalog = catalog
        self.classifier = classifier
        self.quality_engine = quality_engine
    
    def gdpr_assessment(self):
        """Assess GDPR readiness."""
        print(f"
{'='*70}")
        print(f"  📋 GDPR COMPLIANCE ASSESSMENT")
        print(f"{'='*70}")
        
        checks = []
        
        # 1. PII Identification
        pii_cols = [c for c in self.classifier.classifications if c['classification'] == 'PII']
        checks.append({
            "requirement": "PII Identification",
            "status": "✅ PASS" if pii_cols else "❌ FAIL",
            "detail": f"{len(pii_cols)} PII columns identified and tagged"
        })
        
        # 2. Data Minimization
        total_cols = len(self.classifier.classifications)
        pii_pct = len(pii_cols) / total_cols * 100 if total_cols > 0 else 0
        checks.append({
            "requirement": "Data Minimization",
            "status": "✅ PASS" if pii_pct < 30 else "⚠️ REVIEW",
            "detail": f"PII is {pii_pct:.1f}% of all columns"
        })
        
        # 3. Access Controls
        checks.append({
            "requirement": "Access Controls (RBAC)",
            "status": "✅ PASS",
            "detail": "Role-based access matrix defined"
        })
        
        # 4. Right to Deletion
        checks.append({
            "requirement": "Right to Deletion",
            "status": "⚠️ PARTIAL",
            "detail": "Delta Lake supports DELETE, but no automated process"
        })
        
        # 5. Data Quality
        quality_score = self.quality_engine.score()
        checks.append({
            "requirement": "Data Quality Controls",
            "status": "✅ PASS" if quality_score >= 80 else "⚠️ REVIEW",
            "detail": f"Overall quality score: {quality_score}%"
        })
        
        # 6. Audit Trail
        checks.append({
            "requirement": "Audit Trail",
            "status": "✅ PASS",
            "detail": "Delta Lake history provides full audit trail"
        })
        
        # 7. Data Lineage
        checks.append({
            "requirement": "Data Lineage",
            "status": "✅ PASS",
            "detail": "Lineage tracked from source to Gold"
        })
        
        # 8. Masking
        checks.append({
            "requirement": "PII Masking",
            "status": "✅ PASS",
            "detail": "Masked views created for analyst access"
        })
        
        # Print report
        passed = sum(1 for c in checks if "PASS" in c["status"])
        total = len(checks)
        
        for check in checks:
            print(f"  {check['status']} {check['requirement']}")
            print(f"       {check['detail']}")
        
        print(f"
  {'─'*50}")
        print(f"  GDPR Readiness: {passed}/{total} checks passed ({passed/total*100:.0f}%)")
        
        if passed < total:
            print(f"
  📝 REMEDIATION NEEDED:")
            for check in checks:
                if "FAIL" in check["status"] or "PARTIAL" in check["status"]:
                    print(f"    • {check['requirement']}: {check['detail']}")
        
        print(f"{'='*70}")
        return passed / total


# Run compliance assessment
assessor = ComplianceAssessor(catalog, classifier, qe)
gdpr_score = assessor.gdpr_assessment()


---
# 📗 Section 9: Executive Summary (Present to Leadership)

In [0]:
# Generate executive summary
print("""
╔══════════════════════════════════════════════════════════════════════╗
║          DATA GOVERNANCE PLATFORM — EXECUTIVE SUMMARY               ║
╚══════════════════════════════════════════════════════════════════════╝

📊 PLATFORM COVERAGE:
""")

# Catalog stats
catalog_count = len(catalog.entries)
total_rows = sum(e['row_count'] for e in catalog.entries if e['row_count'] > 0)
print(f"   • {catalog_count} tables cataloged across 1 database")
print(f"   • {total_rows:,} total records under governance")

# Classification stats
pii_count = sum(1 for c in classifier.classifications if c['classification'] == 'PII')
fin_count = sum(1 for c in classifier.classifications if c['classification'] == 'Financial')
total_cols = len(classifier.classifications)
print(f"   • {total_cols} columns classified")
print(f"   • {pii_count} PII columns identified and protected")
print(f"   • {fin_count} financial columns identified")

# Quality stats
quality_score = qe.score()
print(f"
📈 QUALITY METRICS:")
print(f"   • Overall quality score: {quality_score}%")
print(f"   • {len(qe.rules)} quality rules active")
print(f"   • {sum(1 for r in qe.results if r['passed'])}/{len(qe.results)} rules passing")

# Compliance
print(f"
🔒 COMPLIANCE:")
print(f"   • GDPR readiness: {gdpr_score*100:.0f}%")
print(f"   • PII masking: Implemented (masked views)")
print(f"   • Access control: 5 roles defined")
print(f"   • Audit trail: Delta Lake history")

# Recommendations
print(f"""
📝 RECOMMENDATIONS:
   1. Implement automated right-to-deletion workflow
   2. Add real-time quality monitoring (streaming checks)
   3. Integrate with Unity Catalog for centralized governance
   4. Set up automated compliance reporting (monthly)
   5. Implement data retention automation (VACUUM scheduling)

🎯 NEXT STEPS:
   • Phase 1: Automate quality checks in CI/CD pipeline
   • Phase 2: Integrate PII detection into ingestion layer
   • Phase 3: Deploy Unity Catalog for cross-workspace governance
""")


---
# 🧹 Cleanup

In [0]:
# Cleanup governance tables
cleanup = [
    "techmart_dw.governance_catalog",
    "techmart_dw.governance_classifications",
    "techmart_dw.governance_quality_results"
]

cleanup_views = [
    "techmart_dw.v_customers_masked",
    "techmart_dw.v_payments_masked"
]

for table in cleanup:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {table}")
    except:
        pass

for view in cleanup_views:
    try:
        spark.sql(f"DROP VIEW IF EXISTS {view}")
    except:
        pass

print(f"✅ Cleaned up {len(cleanup)} tables and {len(cleanup_views)} views")

---
# 📗 Enhancement: Governance and Quality Platform

## Focus: Unity Catalog + Great Expectations + Data Contracts

## Architecture

This capstone integrates multiple concepts from the series:

```
Data Sources
    |
    v (Auto Loader / Lakeflow Connect)
Bronze Layer
    |
    v (DLT with expectations)
Silver Layer
    |
    v (dbt models / Spark)
Gold Layer
    |
    v
Serving + Governance
```

## Key Integration Points

| Component | Role |
|-----------|------|
| Unity Catalog | Governance, permissions, lineage |
| DLT Expectations | Quality gates between layers |
| Lakeflow Jobs | Orchestration and scheduling |
| DABs | CI/CD and deployment |
| Data Contracts | Producer-consumer agreements |

## Production Considerations

1. **Idempotency**: Every pipeline step is safe to rerun
2. **Observability**: Every step logs metrics and status
3. **Alerting**: Failures trigger immediate notifications
4. **Rollback**: Git tags enable quick rollback
5. **Documentation**: All tables documented in Unity Catalog

In [0]:
# Governance and Quality Platform -- Integration patterns
print("Governance and Quality Platform")
print("=" * 60)

integration_checklist = {
    "Data Ingestion": [
        "Auto Loader for file sources",
        "Lakeflow Connect for database CDC",
        "Kafka for streaming events",
    ],
    "Data Quality": [
        "DLT expectations at Bronze->Silver boundary",
        "dbt tests at Silver->Gold boundary",
        "Custom quality checks in Gold",
    ],
    "Governance": [
        "Unity Catalog for all tables",
        "Row-level security for sensitive data",
        "Column masking for PII",
        "Lineage tracking enabled",
    ],
    "Orchestration": [
        "Lakeflow Jobs for scheduling",
        "DABs for deployment",
        "GitHub Actions for CI/CD",
    ],
    "Monitoring": [
        "System tables for pipeline metrics",
        "Slack alerts for failures",
        "Quality score dashboard",
    ],
}

for category, items in integration_checklist.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  * {item}")


---
# 📋 Summary

## What You Built

A complete enterprise data governance platform with:

| Component | What it Does |
|---|---|
| **Data Catalog** | Auto-discovers and documents all tables |
| **Classification** | Identifies PII, Financial, Internal, Public data |
| **Quality Engine** | Configurable rules with scoring dashboard |
| **Access Control** | Role-based permission matrix |
| **Lineage Tracking** | Source → target dependency mapping |
| **PII Masking** | Hash, redact, generalize strategies |
| **Data Contracts** | Producer-consumer SLA agreements |
| **Compliance** | GDPR readiness assessment |
| **Monitoring** | Quality trends and alerting |

## Key Governance Principles

1. **Discover** — You can't govern what you don't know about
2. **Classify** — Sensitivity determines protection level
3. **Protect** — Masking, access control, encryption
4. **Monitor** — Continuous quality and compliance checks
5. **Document** — Contracts, lineage, and catalog

## Production Recommendations

- Use **Unity Catalog** for centralized governance on Databricks
- Implement **Predictive Optimization** for automated maintenance
- Deploy **row-level security** for fine-grained access control
- Integrate with **external tools** (Collibra, Atlan, DataHub) for enterprise catalog

## Next Steps
- **Notebook 32:** Capstone — Orchestrated Data Platform
- **Notebook 34:** Unity Catalog Deep Dive

---
*Notebook 31 of the Databricks Data Engineering series*
*Study Order: 38*